# Build FAISS Index — Google Colab

**Qué hace este notebook:**
1. Lee `movies.csv` y `links.csv` desde Drive
2. Descarga los pósters desde TMDB directamente en Colab
3. Construye el índice FAISS con CLIP
4. Descarga `faiss.index` e `index_metadata.csv`

**Antes de ejecutar:**
- Subir a Google Drive solo `data/raw/movies.csv` y `data/raw/links.csv`
- Tener una TMDB API key
- Activar GPU: `Runtime → Change runtime type → T4 GPU`

In [ ]:
!pip install -q faiss-cpu transformers torch torchvision tqdm pandas Pillow requests

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIGURAR ─────────────────────────────────────────────────────────────────
TMDB_API_KEY = ""  # <-- pegá tu API key acá

MOVIES_CSV   = "/content/movies.csv"
LINKS_CSV    = "/content/links.csv"
POSTERS_DIR  = "/content/posters"

INDEX_PATH      = "/content/faiss.index"
INDEX_META_PATH = "/content/index_metadata.csv"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
import os
import threading
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

TMDB_BASE_URL       = "https://api.themoviedb.org/3"
TMDB_IMAGE_BASE_URL = "https://image.tmdb.org/t/p/w500"


def fetch_movie_details(tmdb_id):
    url = f"{TMDB_BASE_URL}/movie/{tmdb_id}"
    try:
        r = requests.get(url, params={"api_key": TMDB_API_KEY, "language": "en-US"}, timeout=10)
        if r.status_code == 200:
            return r.json()
    except requests.RequestException:
        pass
    return {}


def download_poster(poster_path, save_path):
    if not poster_path:
        return False
    try:
        r = requests.get(f"{TMDB_IMAGE_BASE_URL}{poster_path}", timeout=10)
        if r.status_code == 200:
            save_path.write_bytes(r.content)
            return True
    except requests.RequestException:
        pass
    return False


def build_metadata(links_path, movies_path, posters_dir, max_workers=20):
    posters_dir = Path(posters_dir)
    posters_dir.mkdir(parents=True, exist_ok=True)

    links  = pd.read_csv(links_path).dropna(subset=["tmdbId"])
    links["tmdbId"] = links["tmdbId"].astype(int)
    movies = pd.read_csv(movies_path)
    df     = movies.merge(links, on="movieId")

    rate_limiter = threading.Semaphore(max_workers)

    def process_movie(row):
        tmdb_id     = int(row["tmdbId"])
        poster_file = posters_dir / f"{tmdb_id}.jpg"

        with rate_limiter:
            details = fetch_movie_details(tmdb_id)
        if not details:
            return None

        tmdb_poster_path = details.get("poster_path", "")
        if not tmdb_poster_path:
            return None

        if not poster_file.exists():
            with rate_limiter:
                download_poster(tmdb_poster_path, poster_file)

        if not poster_file.exists():
            return None

        return {
            "movieId":        row["movieId"],
            "tmdbId":         tmdb_id,
            "title":          details.get("title", row["title"]),
            "genres":         ", ".join(g["name"] for g in details.get("genres", [])),
            "overview":       details.get("overview", ""),
            "poster_path":    str(poster_file),
            "tmdb_poster_url": f"{TMDB_IMAGE_BASE_URL}{tmdb_poster_path}",
        }

    rows    = [row for _, row in df.iterrows()]
    records = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_movie, row): row for row in rows}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Descargando pósters"):
            result = future.result()
            if result:
                records.append(result)

    metadata = pd.DataFrame(records)
    print(f"\nPelículas con póster descargado: {len(metadata)}")
    return metadata


metadata = build_metadata(LINKS_CSV, MOVIES_CSV, POSTERS_DIR)
metadata.head()

In [ ]:
import torch
import numpy as np
import faiss
from PIL import Image
from transformers import CLIPProcessor, CLIPModel

print(f"GPU disponible: {torch.cuda.is_available()}")

MODEL_ID = "openai/clip-vit-base-patch32"

class CLIPEncoder:
    def __init__(self, model_id=MODEL_ID, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Cargando CLIP en {self.device}...")
        self.model = CLIPModel.from_pretrained(model_id).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(model_id)
        self.model.eval()

    def _get_image_features(self, inputs):
        vision_outputs = self.model.vision_model(pixel_values=inputs["pixel_values"])
        feats = self.model.visual_projection(vision_outputs.pooler_output)
        return feats / feats.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def encode_images_batch(self, image_paths, batch_size=32):
        all_features = []
        for i in tqdm(range(0, len(image_paths), batch_size), desc="Codificando pósters"):
            batch  = image_paths[i : i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch]
            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
            feats  = self._get_image_features(inputs)
            all_features.append(feats.cpu().numpy())
        return np.vstack(all_features)


class MovieIndex:
    def __init__(self, dim=512):
        self.index    = faiss.IndexFlatIP(dim)
        self.metadata = None

    def build(self, embeddings, metadata):
        self.index.add(embeddings.astype(np.float32))
        self.metadata = metadata.reset_index(drop=True)

    def save(self, index_path, metadata_path):
        faiss.write_index(self.index, index_path)
        self.metadata.to_csv(metadata_path, index=False)
        print(f"Índice guardado:   {index_path}")
        print(f"Metadata guardada: {metadata_path}")

In [ ]:
encoder    = CLIPEncoder()
embeddings = encoder.encode_images_batch(metadata["poster_path"].tolist())

idx = MovieIndex(dim=embeddings.shape[1])
idx.build(embeddings, metadata)
idx.save(INDEX_PATH, INDEX_META_PATH)

print(f"\nListo. {len(metadata)} películas indexadas.")

In [ ]:
from google.colab import files

files.download(INDEX_PATH)
files.download(INDEX_META_PATH)